**Generative AI Use**: For the purposes of the assignments, the use of generative AI is subject to the same policies regarding collaboration. Just as with other collaborators, each student must write down the solutions independently of the output of the interaction and the submission should include a note denoting the nature of the collaboration. The use of generative AI tools to substantially complete sections of the assignments is not in line with the spirit of the assignments, and would be a violation of the [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code).
**关于生成式 AI 的使用**：就本作业而言，使用生成式 AI 与其他形式的协作遵循同样的政策。正如对待其他协作者一样，每位学生都必须独立于交互产生的输出来写下自己的解答，并且提交材料中应注明该协作的性质。使用生成式 AI 工具来实质性地完成作业的各个部分，不符合本作业的精神，将构成对[荣誉准则](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)的违反。


In [ ]:
# This mounts your Google Drive to the Colab VM.
# 这一步把你的 Google Drive 挂载到 Colab 虚拟机上。
from google.colab import drive
drive.mount('/content/drive')

# TODO: Enter the foldername in your Drive where you have saved the unzipped
# assignment folder, e.g. 'cs231n/assignments/assignment1/'
# TODO: 在这里填入你 Drive 中存放已解压作业文件夹的那个目录名，
# 例如 'cs231n/assignments/assignment1/'
FOLDERNAME = 'cs231n/assignments/assignment1/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# Now that we've mounted your Drive, this ensures that
# the Python interpreter of the Colab VM can load
# python files from within it.
# Drive 挂载好之后，这一步保证 Colab 虚拟机里的 Python 解释器
# 能够从该目录中加载 python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# This downloads the CIFAR-10 dataset to your Drive
# if it doesn't already exist.
# 如果 CIFAR-10 数据集还不存在，这一步会把它下载到你的 Drive 里。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME


# Softmax Classifier exercise
# Softmax 分类器练习

*Complete and hand in this completed worksheet (including its outputs and any supporting code outside of the worksheet) with your assignment submission. For more details see the [assignments page](http://vision.stanford.edu/teaching/cs231n/assignments.html) on the course website.*
*请把填写完成的这份作业表（包括它的输出，以及作业表之外的任何支撑代码）随作业一并提交。更多细节见课程网站上的[作业页面](http://vision.stanford.edu/teaching/cs231n/assignments.html)。*

In this exercise you will:
在本练习中，你将：
    

- implement a fully-vectorized **loss function** for the Softmax classifier.
- implement the fully-vectorized expression for its **analytic gradient**
- **check your implementation** using numerical gradient
- use a validation set to **tune the learning rate and regularization** strength
- **optimize** the loss function with **SGD**
- **visualize** the final learned weights

- 为 Softmax 分类器实现完全向量化的**损失函数**。
- 实现它的**解析梯度**的完全向量化表达式
- 用数值梯度**检查你的实现**
- 用验证集**调节学习率和正则化**强度
- 用 **SGD** **优化**损失函数
- **可视化**最终学到的权重


In [ ]:
# Run some setup code for this notebook.
# 运行一些为本 notebook 准备的初始化代码。
import random
import numpy as np
from cs231n.data_utils import load_CIFAR10
import matplotlib.pyplot as plt

# This is a bit of magic to make matplotlib figures appear inline in the
# notebook rather than in a new window.
# 这是一点小魔法，让 matplotlib 的图形直接内嵌显示在
# notebook 里，而不是弹出一个新窗口。
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
                                             # 设置图形的默认尺寸
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Some more magic so that the notebook will reload external python modules;
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
# 再一点魔法，让 notebook 能够重新加载外部的 python 模块；
# 参见 http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2


## CIFAR-10 Data Loading and Preprocessing
## CIFAR-10 数据加载与预处理


In [ ]:
# Load the raw CIFAR-10 data.
# 加载原始的 CIFAR-10 数据。
cifar10_dir = 'cs231n/datasets/cifar-10-batches-py'

# Cleaning up variables to prevent loading data multiple times (which may cause memory issue)
# 清理变量，避免多次重复加载数据（重复加载可能导致内存问题）
try:
   del X_train, y_train
   del X_test, y_test
   print('Clear previously loaded data.')
except:
   pass

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

# As a sanity check, we print out the size of the training and test data.
# 作为一次合理性检查，我们打印出训练数据和测试数据的大小。
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)


In [ ]:
# Visualize some examples from the dataset.
# We show a few examples of training images from each class.
# 可视化数据集中的一些样本。
# 我们从每个类别中展示若干张训练图像作为例子。
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(classes)
samples_per_class = 7
for y, cls in enumerate(classes):
    idxs = np.flatnonzero(y_train == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + y + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X_train[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls)
plt.show()


In [ ]:
# Split the data into train, val, and test sets. In addition we will
# create a small development set as a subset of the training data;
# we can use this for development so our code runs faster.
# 把数据划分成训练集、验证集和测试集。此外，
# 我们还会创建一个小的开发集（development set），它是训练数据的一个子集；
# 有了它，开发时我们的代码能跑得更快。
num_training = 49000
num_validation = 1000
num_test = 1000
num_dev = 500

# Our validation set will be num_validation points from the original
# training set.
# 我们的验证集由原训练集中的 num_validation 个点组成。
mask = range(num_training, num_training + num_validation)
X_val = X_train[mask]
y_val = y_train[mask]

# Our training set will be the first num_train points from the original
# training set.
# 我们的训练集由原训练集中的前 num_training 个点组成。
mask = range(num_training)
X_train = X_train[mask]
y_train = y_train[mask]

# We will also make a development set, which is a small subset of
# the training set.
# 我们还会做一个开发集，它是训练集的一个小子集。
mask = np.random.choice(num_training, num_dev, replace=False)
X_dev = X_train[mask]
y_dev = y_train[mask]

# We use the first num_test points of the original test set as our
# test set.
# 我们用原测试集的前 num_test 个点作为测试集。
mask = range(num_test)
X_test = X_test[mask]
y_test = y_test[mask]

print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)


In [ ]:
# Preprocessing: reshape the image data into rows
# 预处理：把图像数据重塑成一行一个样本
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_val = np.reshape(X_val, (X_val.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))
X_dev = np.reshape(X_dev, (X_dev.shape[0], -1))

# As a sanity check, print out the shapes of the data
# 作为合理性检查，打印出各个数据的形状
print('Training data shape: ', X_train.shape)
print('Validation data shape: ', X_val.shape)
print('Test data shape: ', X_test.shape)
print('dev data shape: ', X_dev.shape)


In [ ]:
# Preprocessing: subtract the mean image
# 预处理：减去均值图像
# first: compute the image mean based on the training data
# 第一步：基于训练数据计算图像均值
mean_image = np.mean(X_train, axis=0)
print(mean_image[:10]) # print a few of the elements
                       # 打印其中几个元素
plt.figure(figsize=(4,4))
plt.imshow(mean_image.reshape((32,32,3)).astype('uint8')) # visualize the mean image
                                                          # 可视化均值图像
plt.show()

# second: subtract the mean image from train and test data
# 第二步：从训练数据和测试数据中减去均值图像
X_train -= mean_image
X_val -= mean_image
X_test -= mean_image
X_dev -= mean_image

# third: append the bias dimension of ones (i.e. bias trick) so that our classifier
# only has to worry about optimizing a single weight matrix W.
# 第三步：追加一维全为 1 的偏置维度（也就是 bias trick），
# 这样我们的分类器只需操心优化单个权重矩阵 W。
X_train = np.hstack([X_train, np.ones((X_train.shape[0], 1))])
X_val = np.hstack([X_val, np.ones((X_val.shape[0], 1))])
X_test = np.hstack([X_test, np.ones((X_test.shape[0], 1))])
X_dev = np.hstack([X_dev, np.ones((X_dev.shape[0], 1))])

print(X_train.shape, X_val.shape, X_test.shape, X_dev.shape)


## Softmax Classifier
## Softmax 分类器

Your code for this section will all be written inside `cs231n/classifiers/softmax.py`.
本节的所有代码都写在 `cs231n/classifiers/softmax.py` 里面。

As you can see, we have prefilled the function `softmax_loss_naive` which uses for loops to evaluate the softmax loss function.
可以看到，我们已经预先填好了函数 `softmax_loss_naive`，它用 for 循环来计算 softmax 损失函数。


In [ ]:
# Evaluate the naive implementation of the loss we provided for you:
# 评估我们为你提供的朴素损失实现：
from cs231n.classifiers.softmax import softmax_loss_naive
import time

# generate a random Softmax classifier weight matrix of small numbers
# 生成一个数值很小的随机 Softmax 分类器权重矩阵
W = np.random.randn(3073, 10) * 0.0001

loss, grad = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
print('loss: %f' % (loss, ))

# As a rough sanity check, our loss should be something close to -log(0.1).
# 做一个粗略的合理性检查：损失应该接近 -log(0.1)。
print('loss: %f' % loss)
print('sanity check: %f' % (-np.log(0.1)))


**Inline Question 1**
**内嵌问题 1**

Why do we expect our loss to be close to -log(0.1)? Explain briefly.**
为什么我们期望损失接近 -log(0.1)？简要说明。**

$\color{blue}{\textit Your Answer:}$ *Fill this in*
**你的答案：** *在这里填写*



The `grad` returned from the function above is right now all zero. Derive and implement the gradient for the softmax loss function and implement it inline inside the function `softmax_loss_naive`. You will find it helpful to interleave your new code inside the existing function.
上面这个函数返回的 `grad` 现在全是零。请推导 softmax 损失函数的梯度，并把它就地实现在函数 `softmax_loss_naive` 内部。把新代码穿插进现有函数里会对你有帮助。

To check that you have correctly implemented the gradient, you can numerically estimate the gradient of the loss function and compare the numeric estimate to the gradient that you computed. We have provided code that does this for you:
要检查你实现的梯度是否正确，你可以数值地估计损失函数的梯度，再把它与你算出的梯度比较。我们已经为你提供了做这件事的代码：


In [ ]:
# Once you've implemented the gradient, recompute it with the code below
# and gradient check it with the function we provided for you
# 实现完梯度之后，用下面的代码重新计算它，
# 并用我们提供的函数对它做梯度检查

# Compute the loss and its gradient at W.
# 在 W 处计算损失及其梯度。
loss, grad = softmax_loss_naive(W, X_dev, y_dev, 0.0)

# Numerically compute the gradient along several randomly chosen dimensions, and
# compare them with your analytically computed gradient. The numbers should match
# almost exactly along all dimensions.
# 在若干随机选取的维度上数值地估计梯度，
# 并把它们与你解析计算出的梯度比较。在所有维度上这两个数应该几乎完全一致。
from cs231n.gradient_check import grad_check_sparse
f = lambda w: softmax_loss_naive(w, X_dev, y_dev, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, grad)

# do the gradient check once again with regularization turned on
# you didn't forget the regularization gradient did you?
# 把正则化打开，再做一次梯度检查
# 你没忘记正则化项的梯度吧？
loss, grad = softmax_loss_naive(W, X_dev, y_dev, 5e1)
f = lambda w: softmax_loss_naive(w, X_dev, y_dev, 5e1)[0]
grad_numerical = grad_check_sparse(f, W, grad)


**Inline Question 2**
**内嵌问题 2**

Although gradcheck is reliable softmax loss, it is possible that for SVM loss, once in a while, a dimension in the gradcheck will not match exactly. What could such a discrepancy be caused by? Is it a reason for concern? What is a simple example in one dimension where a svm loss gradient check could fail? How would change the margin affect of the frequency of this happening?
尽管梯度检查对 softmax 损失是可靠的，但对 SVM 损失而言，偶尔会出现梯度检查中某个维度不完全匹配的情况。这种偏差可能是由什么引起的？它值得担心吗？在一维情形下给出一个 SVM 损失梯度检查可能失败的简单例子。改变间隔（margin）会如何影响这种情况的发生频率？

Note that SVM loss for a sample $(x_i, y_i)$ is defined as: $$L_i = \sum_{j\ne y_i}\max(0, s_j - s_{y_i} + \Delta)$$ where $j$ iterates over all classes except the correct class $y_i$ and $s_j$ denotes the classifier score for $j^{th}$ class. $\Delta$ is a scalar margin. For more information, refer to 'Multiclass Support Vector Machine loss' on [this](https://cs231n.github.io/linear-classify/) page.
注意，样本 $(x_i, y_i)$ 的 SVM 损失定义为：$$L_i = \sum_{j\ne y_i}\max(0, s_j - s_{y_i} + \Delta)$$ 其中 $j$ 遍历除正确类别 $y_i$ 之外的所有类别，$s_j$ 表示第 $j^{th}$ 个类别的分类器得分。$\Delta$ 是一个标量间隔（margin）。更多信息请参考[这个](https://cs231n.github.io/linear-classify/)页面上的 'Multiclass Support Vector Machine loss'。

*Hint: the SVM loss function is not strictly speaking differentiable.*
*提示：严格来说，SVM 损失函数并不可微。*


$\color{blue}{\textit Your Answer:}$ *fill this in.*  
**你的答案：** *在这里填写。*



In [ ]:
# Next implement the function softmax_loss_vectorized; for now only compute the loss;
# we will implement the gradient in a moment.
# 接下来实现函数 softmax_loss_vectorized；眼下只计算损失，
# 梯度稍后再实现。
tic = time.time()
loss_naive, grad_naive = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Naive loss: %e computed in %fs' % (loss_naive, toc - tic))

from cs231n.classifiers.softmax import softmax_loss_vectorized
tic = time.time()
loss_vectorized, _ = softmax_loss_vectorized(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Vectorized loss: %e computed in %fs' % (loss_vectorized, toc - tic))

# The losses should match but your vectorized implementation should be much faster.
# 两个损失应该一致，但你的向量化实现应该快得多。
print('difference: %f' % (loss_naive - loss_vectorized))


In [ ]:
# Complete the implementation of softmax_loss_vectorized, and compute the gradient
# of the loss function in a vectorized way.
# 完成 softmax_loss_vectorized 的实现，并以向量化的方式
# 计算损失函数的梯度。

# The naive implementation and the vectorized implementation should match, but
# the vectorized version should still be much faster.
# 朴素实现与向量化实现的结果应该一致，
# 但向量化版本仍然应该快得多。
tic = time.time()
_, grad_naive = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Naive loss and gradient: computed in %fs' % (toc - tic))

tic = time.time()
_, grad_vectorized = softmax_loss_vectorized(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Vectorized loss and gradient: computed in %fs' % (toc - tic))

# The loss is a single number, so it is easy to compare the values computed
# by the two implementations. The gradient on the other hand is a matrix, so
# we use the Frobenius norm to compare them.
# 损失是一个标量，所以比较两个实现算出的值很容易。
# 而梯度是一个矩阵，所以我们用 Frobenius 范数来比较它们。
difference = np.linalg.norm(grad_naive - grad_vectorized, ord='fro')
print('difference: %f' % difference)


### Stochastic Gradient Descent
### 随机梯度下降

We now have vectorized and efficient expressions for the loss, the gradient and our gradient matches the numerical gradient. We are therefore ready to do SGD to minimize the loss. Your code for this part will be written inside `cs231n/classifiers/linear_classifier.py`.
现在我们已经有了向量化且高效的损失与梯度表达式，而且梯度与数值梯度一致。因此可以开始用 SGD 最小化损失了。这一部分的代码写在 `cs231n/classifiers/linear_classifier.py` 里面。


In [ ]:
# In the file linear_classifier.py, implement SGD in the function
# LinearClassifier.train() and then run it with the code below.
# 在文件 linear_classifier.py 中，在函数 LinearClassifier.train() 里实现 SGD，
# 然后用下面的代码运行它。
from cs231n.classifiers import Softmax
softmax = Softmax()
tic = time.time()
loss_hist = softmax.train(X_train, y_train, learning_rate=1e-7, reg=2.5e4,
                      num_iters=1500, verbose=True)
toc = time.time()
print('That took %fs' % (toc - tic))


In [ ]:
# A useful debugging strategy is to plot the loss as a function of
# iteration number:
# 一个有用的调试策略是把损失画成迭代次数的函数：
plt.plot(loss_hist)
plt.xlabel('Iteration number')
plt.ylabel('Loss value')
plt.show()


In [ ]:
# Write the LinearClassifier.predict function and evaluate the performance on
# both the training and validation set
# You should get validation accuracy of about 0.34 (> 0.33).
# 编写 LinearClassifier.predict 函数，并在训练集和验证集上
# 评估它的表现
# 你应该能得到约 0.34（> 0.33）的验证准确率。
y_train_pred = softmax.predict(X_train)
print('training accuracy: %f' % (np.mean(y_train == y_train_pred), ))
y_val_pred = softmax.predict(X_val)
print('validation accuracy: %f' % (np.mean(y_val == y_val_pred), ))


In [ ]:
# Save the trained model for autograder.
# 保存训练好的模型，供自动评分器（autograder）使用。
softmax.save("softmax.npy")


In [ ]:
# Use the validation set to tune hyperparameters (regularization strength and
# learning rate). You should experiment with different ranges for the learning
# rates and regularization strengths; if you are careful you should be able to
# get a classification accuracy of about 0.365 (> 0.36) on the validation set.
# 用验证集来调节超参数（正则化强度和
# 学习率）。你应该多试几个不同的学习率范围和
# 正则化强度范围；如果做得仔细，你应该能在验证集上
# 得到约 0.365（> 0.36）的分类准确率。

# Note: you may see runtime/overflow warnings during hyper-parameter search.
# This may be caused by extreme values, and is not a bug.
# 注意：在搜索超参数的过程中你可能会看到运行时/溢出警告。
# 这可能是极端数值造成的，不是 bug。

# results is dictionary mapping tuples of the form
# (learning_rate, regularization_strength) to tuples of the form
# (training_accuracy, validation_accuracy). The accuracy is simply the fraction
# of data points that are correctly classified.
# results 是一个字典，把形如
# (learning_rate, regularization_strength) 的元组映射到形如
# (training_accuracy, validation_accuracy) 的元组。准确率就是
# 被正确分类的数据点所占的比例。
results = {}
best_val = -1   # The highest validation accuracy that we have seen so far.
                 # 目前为止见到过的最高验证准确率。
best_softmax = None # The Softmax object that achieved the highest validation rate.
                    # 取得最高验证准确率的那个 Softmax 对象。

################################################################################
# TODO:                                                                        #
# Write code that chooses the best hyperparameters by tuning on the validation #
# set. For each combination of hyperparameters, train a Softmax on the.        #
# training set, compute its accuracy on the training and validation sets, and  #
# store these numbers in the results dictionary. In addition, store the best   #
# validation accuracy in best_val and the Softmax object that achieves this.   #
# accuracy in best_softmax.                                                    #
#                                                                              #
# Hint: You should use a small value for num_iters as you develop your         #
# validation code so that the classifiers don't take much time to train; once  #
# you are confident that your validation code works, you should rerun the      #
# code with a larger value for num_iters.                                      #
# TODO:                                                                        #
# 写代码在验证集上调节，选出最佳超参数。                                                          #
# 对每一组超参数，在训练集上训练一个 Softmax，                                                   #
# 计算它在训练集和验证集上的准确率，                                                            #
# 并把这些数存进 results 字典。此外，把最好的验证准确率                                              #
# 存到 best_val，把达到该准确率的 Softmax 对象存到 best_softmax。                              #
#                                                                              #
# 提示：在开发验证代码时应该用较小的 num_iters，                                                 #
# 这样分类器训练起来不会太费时间；                                                             #
# 等你确信验证代码没问题之后，再用更大的 num_iters 重新跑一遍。                                         #
################################################################################

# Provided as a reference. You may or may not want to change these hyperparameters
# 此处仅供参考，这些超参数你可能想改，也可能不想改
learning_rates = [1e-7, 1e-6]
regularization_strengths = [2.5e4, 1e4]



# Print out results.
# 打印结果。
for lr, reg in sorted(results):
    train_accuracy, val_accuracy = results[(lr, reg)]
    print('lr %e reg %e train accuracy: %f val accuracy: %f' % (
                lr, reg, train_accuracy, val_accuracy))

print('best validation accuracy achieved during cross-validation: %f' % best_val)

In [ ]:
# Visualize the cross-validation results
# 可视化交叉验证的结果
import math
import pdb

# pdb.set_trace()

x_scatter = [math.log10(x[0]) for x in results]
y_scatter = [math.log10(x[1]) for x in results]

# plot training accuracy
# 画出训练准确率
marker_size = 100
colors = [results[x][0] for x in results]
plt.subplot(2, 1, 1)
plt.tight_layout(pad=3)
plt.scatter(x_scatter, y_scatter, marker_size, c=colors, cmap=plt.cm.coolwarm)
plt.colorbar()
plt.xlabel('log learning rate')
plt.ylabel('log regularization strength')
plt.title('CIFAR-10 training accuracy')

# plot validation accuracy
# 画出验证准确率
colors = [results[x][1] for x in results] # default size of markers is 20
                                          # 标记的默认大小是 20
plt.subplot(2, 1, 2)
plt.scatter(x_scatter, y_scatter, marker_size, c=colors, cmap=plt.cm.coolwarm)
plt.colorbar()
plt.xlabel('log learning rate')
plt.ylabel('log regularization strength')
plt.title('CIFAR-10 validation accuracy')
plt.show()


In [ ]:
# Evaluate the best softmax on test set
# 在测试集上评估最好的 softmax
y_test_pred = best_softmax.predict(X_test)
test_accuracy = np.mean(y_test == y_test_pred)
print('Softmax classifier on raw pixels final test set accuracy: %f' % test_accuracy)


In [ ]:
# Save best softmax model
# 保存最好的 softmax 模型
best_softmax.save("best_softmax.npy")


In [ ]:
# Visualize the learned weights for each class.
# Depending on your choice of learning rate and regularization strength, these may
# or may not be nice to look at.
# 可视化每个类别学到的权重。
# 取决于你选择的学习率和正则化强度，
# 这些图可能好看，也可能不好看。
w = best_softmax.W[:-1,:] # strip out the bias
                          # 去掉偏置
w = w.reshape(32, 32, 3, 10)
w_min, w_max = np.min(w), np.max(w)
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
for i in range(10):
    plt.subplot(2, 5, i + 1)

    # Rescale the weights to be between 0 and 255
    # 把权重重新缩放到 0 到 255 之间
    wimg = 255.0 * (w[:, :, :, i].squeeze() - w_min) / (w_max - w_min)
    plt.imshow(wimg.astype('uint8'))
    plt.axis('off')
    plt.title(classes[i])


**Inline question 3**
**内嵌问题 3**

Describe what your visualized Softmax classifier weights look like, and offer a brief explanation for why they look the way they do.
描述你可视化出来的 Softmax 分类器权重长什么样，并简要解释它们为什么是这个样子。

$\color{blue}{\textit Your Answer:}$ *fill this in*  
**你的答案：** *在这里填写*


**Inline Question 4** - *True or False*
**内嵌问题 4** - *判断对错*

Suppose the overall training loss is defined as the sum of the per-datapoint loss over all training examples. It is possible to add a new datapoint to a training set that would change the softmax loss, but leave the SVM loss unchanged.
假设整体训练损失定义为所有训练样本上各数据点损失之和。有可能向训练集中加入一个新的数据点，使得 softmax 损失改变，而 SVM 损失保持不变。

$\color{blue}{\textit Your Answer:}$
**你的答案：**


$\color{blue}{\textit Your Explanation:}$
**你的解释：**

